# LeetCode 640 — Solve the Equation bằng Python

Link đề: https://leetcode.com/problems/solve-the-equation/ · Độ khó: **Medium** · Chủ đề: Math, String, Linear Algebra, Simulation

Skeleton LeetCode:

```python
class Solution:
    def solveEquation(self, equation: str) -> str:
        ...
```

Ghi chú: ví dụ 1-3 lấy từ đề; các ví dụ khác do notebook tự đặt (đáp án đã kiểm chứng bằng code độc lập), cần thì đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: giải phương trình bậc nhất một ẩn `x` cho dưới dạng chuỗi (chỉ có `+`, `-`, số nguyên, ẩn `x` và dấu `=`), trả về `"x=<giá trị>"`, hoặc `"No solution"`, hoặc `"Infinite solutions"`.
2. Nắm các công cụ Python cần dùng: duyệt chuỗi bằng chỉ số, `split("=")`, `str.isdigit`, đọc số có dấu bằng con trỏ, chia nguyên `//` so với `/`, `re.findall`, `eval` có kiểm soát, số phức `1j`, và `Fraction`.
3. Đi từ **thử từng giá trị `x`** -> **quy về `a*x + b = 0` bằng cách duyệt từng vế** -> **một lượt duyệt, đổi dấu khi qua dấu `=`** -> **nội suy từ hai điểm `f(0)`, `f(1)`** -> **mẹo thay `x` bằng số phức `i` để "tự tách" hệ số và hằng số**.
4. Hiểu hai mấu chốt: (a) mọi biểu thức ở đây đều **tuyến tính** theo `x` nên đủ **hai số** `(hệ số của x, hằng số)` để mô tả một vế; (b) ba kết cục `x=...`, `No solution`, `Infinite solutions` chỉ phụ thuộc `a` (hệ số) và `b` (hằng số) sau khi chuyển vế.
5. Tránh các lỗi kinh điển: hiểu sai `x` đứng một mình (hệ số 1) và `-x`, quên đổi dấu khi chuyển vế, chia số thực (`"x=2.0"`), chia cho `0`, nhầm `No solution` với `Infinite solutions`, và tách chuỗi sai chỗ (`split("+")`).

## 2. Đề bài & ví dụ

Giải một phương trình cho trước và trả về giá trị của `x` dưới dạng chuỗi `"x=#value"`. Phương trình chỉ chứa phép `+`, `-`, biến `x` và hệ số của nó. Trả về `"No solution"` nếu phương trình **vô nghiệm**, hoặc `"Infinite solutions"` nếu **vô số nghiệm**.

Nếu phương trình có **đúng một nghiệm**, đề đảm bảo giá trị của `x` là **số nguyên**.

**Ví dụ 1**

- Input: `equation = "x+5-3+x=6+x-2"`
- Output: `"x=2"`

**Ví dụ 2**

- Input: `equation = "x=x"`
- Output: `"Infinite solutions"`

**Ví dụ 3**

- Input: `equation = "2x=x"`
- Output: `"x=0"`

**Ràng buộc:**

- `3 <= equation.length <= 1000`
- `equation` có **đúng một** dấu `=`.
- `equation` gồm các số nguyên có giá trị tuyệt đối trong `[0, 100]` (không có số 0 ở đầu) và biến `x`.
- Nếu có đúng một nghiệm thì nó là số nguyên.

Chú ý quan trọng:

- Hệ số `1` bị **lược bỏ**: `x` nghĩa là `1x`, `-x` nghĩa là `-1x`. Hệ số `0` được viết rõ: `0x`.
- Các số hạng nối nhau bằng `+` hoặc `-`; số hạng đầu tiên của mỗi vế có thể có dấu `-` ở đầu.
- Kết quả `x=0` viết là `"x=0"` (không phải `"x=-0"` hay `"x=0.0"`).

In [1]:
import re
import random
import time
from fractions import Fraction

examples = [
    ("x+5-3+x=6+x-2", "x=2"),
    ("x=x", "Infinite solutions"),
    ("2x=x", "x=0"),
]
for eq, expected in examples:
    print(f"{eq!r:20} -> kỳ vọng {expected!r}")

'x+5-3+x=6+x-2'      -> kỳ vọng 'x=2'
'x=x'                -> kỳ vọng 'Infinite solutions'
'2x=x'               -> kỳ vọng 'x=0'


## 3. Kiến thức nền

### 3.1. Phương trình bậc nhất `a*x + b = 0` và ba kết cục

Chuyển mọi số hạng về vế trái ta được `a*x + b = 0` với `a` là tổng các hệ số của `x`, `b` là tổng các hằng số.

- Nếu `a != 0`: **đúng một nghiệm** `x = -b / a`.
- Nếu `a == 0` và `b == 0`: phương trình thành `0 = 0`, đúng với **mọi** `x` -> **vô số nghiệm**.
- Nếu `a == 0` và `b != 0`: thành `b = 0` với `b != 0`, **không** `x` nào thỏa -> **vô nghiệm**.

Ví dụ: `2x + 2 = x + 4`. Chuyển vế: `(2x - x) + (2 - 4) = 0` tức `x - 2 = 0`, `a = 1, b = -2`, nghiệm `x = 2`.

In [2]:
def classify(a, b):
    if a != 0:
        return f"x={-b // a}"
    return "Infinite solutions" if b == 0 else "No solution"

print(classify(1, -2))     # x - 2 = 0
print(classify(0, 0))      # 0 = 0
print(classify(0, 7))      # 7 = 0 (sai)
print(classify(-2, 6))     # -2x + 6 = 0 -> x = 3

x=2
Infinite solutions
No solution
x=3


### 3.2. Duyệt chuỗi, `split`, `isdigit`

- `s.split("=")` cắt chuỗi tại dấu `=` thành list: `"2x=x".split("=")` = `["2x", "x"]`. Vì đề đảm bảo **đúng một** dấu `=`, ta luôn nhận đúng hai vế.
- `s[i]` lấy ký tự thứ `i`; `c.isdigit()` kiểm tra ký tự là chữ số. Chú ý `"-".isdigit()` và `"x".isdigit()` đều là `False`.
- Có thể gán hai vế luôn: `left, right = equation.split("=")`.

In [3]:
eq = "x+5-3+x=6+x-2"
left, right = eq.split("=")
print(repr(left), repr(right))
print([c for c in left])
print([c.isdigit() for c in left])

'x+5-3+x' '6+x-2'
['x', '+', '5', '-', '3', '+', 'x']
[False, False, True, False, True, False, False]


### 3.3. Đọc một số nguyên bằng con trỏ, và hệ số bị lược

Đứng ở vị trí `i` (ký tự đầu của một số hạng, **không** phải dấu). Cho `j` chạy tiếp qua các chữ số. Có hai tình huống:

- Sau các chữ số là `x` -> số hạng chứa `x`. Nếu **không có chữ số nào** (`j == i`) thì hệ số là `1`.
- Ngược lại -> số hạng là một hằng số.

In [4]:
def read_term(s, i):
    """Đọc số hạng không dấu bắt đầu ở s[i]. Trả về (loại, giá trị, vị trí kế tiếp)."""
    j = i
    while j < len(s) and s[j].isdigit():
        j += 1
    if j < len(s) and s[j] == "x":
        coef = int(s[i:j]) if j > i else 1           # "x" nghĩa là 1x
        return "x", coef, j + 1
    return "const", int(s[i:j]), j

print(read_term("12x+5", 0))     # ('x', 12, 3)
print(read_term("x+5", 0))       # ('x', 1, 1)
print(read_term("0x=3", 0))      # ('x', 0, 2)
print(read_term("5-3", 0))       # ('const', 5, 1)
print(read_term("100", 0))       # ('const', 100, 3)

('x', 12, 3)
('x', 1, 1)
('x', 0, 2)
('const', 5, 1)
('const', 100, 3)


### 3.4. Dấu của từng số hạng

Dấu `+` hoặc `-` đứng **trước** số hạng của nó. Ta nhớ biến `sign` (`+1` hoặc `-1`): gặp `+` đặt `sign = 1`, gặp `-` đặt `sign = -1`, sau khi đọc xong một số hạng đặt lại `sign = 1` (số hạng kế tiếp nếu không có dấu sẽ là dương, ví dụ ở đầu vế).

In [5]:
def terms_with_sign(side):
    out, sign, i = [], 1, 0
    while i < len(side):
        if side[i] == "+":
            sign, i = 1, i + 1
        elif side[i] == "-":
            sign, i = -1, i + 1
        else:
            kind, val, i = read_term(side, i)
            out.append((kind, sign * val))
            sign = 1
    return out

print(terms_with_sign("x+5-3+x"))
print(terms_with_sign("-x+0x-10"))

[('x', 1), ('const', 5), ('const', -3), ('x', 1)]
[('x', -1), ('x', 0), ('const', -10)]


### 3.5. Chia nguyên `//` và chia thực `/`

`/` luôn trả về **số thực** (`6 / 3 = 2.0`), còn `//` trả về **chia nguyên làm tròn xuống** (`6 // 3 = 2`). Đề cần `"x=2"` chứ không phải `"x=2.0"`, và kết quả chia **chính xác** (nghiệm nguyên) nên `//` cho đúng giá trị. Chú ý với số âm `//` làm tròn về `-vô cực` (`-7 // 2 = -4`), nhưng khi phép chia hết thì không có chuyện làm tròn.

In [6]:
print(6 / 3, 6 // 3, -6 // 3, 6 // -3, -6 // -3)
print(-7 // 2, 7 // 2)                # chỉ khác biệt khi không chia hết
print(0 / -3, 0 // -3)                # -0.0 và 0: / có thể sinh "-0.0"
print(f"x={6 // 3}", f"x={6 / 3}")

2.0 2 -2 -2 2
-4 3
-0.0 0
x=2 x=2.0


### 3.6. Biểu thức chính quy: tách số hạng bằng `re.findall`

Mẫu `[+-]?\d*x|[+-]?\d+` tìm các số hạng: một dấu tùy chọn, rồi (một dãy chữ số tùy chọn và chữ `x`) **hoặc** (một dãy chữ số). `re.findall` trả về list các số hạng khớp, từ trái sang phải.

In [7]:
pattern = re.compile(r"[+-]?\d*x|[+-]?\d+")
print(pattern.findall("x+5-3+x"))
print(pattern.findall("-x+0x-10"))
print(pattern.findall("100x-100"))

['x', '+5', '-3', '+x']
['-x', '+0x', '-10']
['100x', '-100']


### 3.7. `eval` có kiểm soát và số phức `1j`

`eval(chuỗi)` tính một biểu thức Python viết dưới dạng chuỗi: `eval("2+3*4")` = `14`. Vì `eval` có thể chạy mã tùy ý, **chỉ** dùng với dữ liệu đã kiểm tra (ở đây chuỗi chỉ gồm chữ số, `x`, `+`, `-`, `=`) và nên tắt các hàm dựng sẵn bằng đối số `{"__builtins__": {}}`.

Python có số phức: `1j` là đơn vị ảo; `3 + 2j` có `.real = 3.0` và `.imag = 2.0`. Mẹo của cách 5: **thay `x` bằng `1j`**: khi đó phần **ảo** gom **hệ số của `x`**, phần **thực** gom **hằng số** (vì cộng/trừ số phức tách riêng thực và ảo, và `2x` thành `2j`).

In [8]:
print(eval("2+3*4"), eval("x+1", {"__builtins__": {}}, {"x": 10}))
z = eval("j+5-3+j", {"__builtins__": {}}, {"j": 1j})     # "x+5-3+x" với x -> j
print(z, "| thực =", z.real, "(hằng số) | ảo =", z.imag, "(hệ số của x)")
print(eval("2j-10", {"__builtins__": {}}, {}), "<- '2x-10' khi x -> j")

14 11
(2+2j) | thực = 2.0 (hằng số) | ảo = 2.0 (hệ số của x)
(-10+2j) <- '2x-10' khi x -> j


### 3.8. Hàm tuyến tính được xác định bởi hai điểm

Nếu `f(x) = a*x + b` thì `f(0) = b` và `f(1) = a + b`, suy ra `a = f(1) - f(0)`. Chỉ cần **tính giá trị biểu thức tại hai điểm** là biết `a` và `b`, mà không cần tự phân tích chuỗi (cách 4).

### 3.9. `Fraction` và hàm `abs`

`Fraction(p, q)` là phân số chính xác (không làm tròn). Dùng ở phần biến thể để xử lý nghiệm không nguyên.

In [9]:
f = lambda x: 3 * x - 7
a = f(1) - f(0)
b = f(0)
print("a =", a, "b =", b)
print(Fraction(3, 6), Fraction(-4, 2), Fraction(7, 2).numerator, Fraction(7, 2).denominator)

a = 3 b = -7
1/2 -2 7 2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Giải bằng tay `x+5-3+x=6+x-2`:

1. Vế trái: các số hạng chứa `x`: `x`, `x` -> `2x`. Hằng số: `+5`, `-3` -> `2`. Vế trái = `2x + 2`.
2. Vế phải: `x` -> `1x`; hằng số `6`, `-2` -> `4`. Vế phải = `x + 4`.
3. `2x + 2 = x + 4`. Chuyển `x` sang trái, hằng số sang phải: `2x - x = 4 - 2` -> `x = 2`.

Công thức: với vế trái `a1*x + b1` và vế phải `a2*x + b2`:

```
(a1 - a2) * x = b2 - b1
```

Bảng tổng hợp (mỗi dòng là một số hạng):

| Vế | Số hạng | Hệ số `x` | Hằng số |
|---|---|---|---|
| trái | `x` | 1 | 0 |
| trái | `+5` | 0 | 5 |
| trái | `-3` | 0 | -3 |
| trái | `+x` | 1 | 0 |
| **trái tổng** | | **2** | **2** |
| phải | `6` | 0 | 6 |
| phải | `+x` | 1 | 0 |
| phải | `-2` | 0 | -2 |
| **phải tổng** | | **1** | **4** |

`a1 - a2 = 2 - 1 = 1`, `b2 - b1 = 4 - 2 = 2`, nên `x = 2 / 1 = 2`. Ba ví dụ của đề:

| Phương trình | `a1, b1` | `a2, b2` | `a1 - a2` | `b2 - b1` | Kết luận |
|---|---|---|---|---|---|
| `x=x` | 1, 0 | 1, 0 | 0 | 0 | vô số nghiệm |
| `2x=x` | 2, 0 | 1, 0 | 1 | 0 | `x = 0` |
| `x+1=x` (tự đặt) | 1, 1 | 1, 0 | 0 | -1 | vô nghiệm |

## 5. Cách 1 — Brute Force / baseline: thử từng giá trị `x`

**Ý tưởng.** Chưa cần đại số: viết hàm `eval_side(side, x)` tính **giá trị một vế** khi thay `x` bằng một số cụ thể, rồi **thử mọi `x` trong một khoảng** và xem `x` nào làm hai vế bằng nhau.

Cần biết thử trong khoảng nào: nghiệm (nếu có) thỏa `|x| = |b2 - b1| / |a1 - a2| <= tổng giá trị tuyệt đối mọi số trong phương trình` (vì `|a1 - a2| >= 1` và `|b2 - b1|` không vượt tổng đó). Đặt `B = tổng mọi số + 1` và thử `x` từ `-B` đến `B` (ít nhất 3 giá trị).

Số `x` làm hai vế bằng nhau cho ta biết kết cục: **không có** -> vô nghiệm; **đúng một** -> `x` đó; **tất cả** -> vô số nghiệm (vì hàm tuyến tính nên chỉ có ba khả năng này).

In [10]:
def eval_side(side, x):
    """Giá trị của một vế khi thay x bằng số cụ thể."""
    total, sign, i = 0, 1, 0
    while i < len(side):
        if side[i] == "+":
            sign, i = 1, i + 1
        elif side[i] == "-":
            sign, i = -1, i + 1
        else:
            kind, val, i = read_term(side, i)
            total += sign * val * (x if kind == "x" else 1)
            sign = 1
    return total

print(eval_side("x+5-3+x", 0), eval_side("x+5-3+x", 1), eval_side("x+5-3+x", 10))   # 2, 4, 22
print(eval_side("6+x-2", 2))                                                        # 6

def solve_brute(equation):
    left, right = equation.split("=")
    B = sum(int(t) for t in re.findall(r"\d+", equation)) + 1
    roots = [x for x in range(-B, B + 1) if eval_side(left, x) == eval_side(right, x)]
    if not roots:
        return "No solution"
    if len(roots) == 2 * B + 1:
        return "Infinite solutions"
    return f"x={roots[0]}"

for eq, expected in examples:
    print(f"{eq!r:20} -> {solve_brute(eq)!r}", "OK" if solve_brute(eq) == expected else "SAI")

2 4 22
6
'x+5-3+x=6+x-2'      -> 'x=2' OK
'x=x'                -> 'Infinite solutions' OK
'2x=x'               -> 'x=0' OK


In [11]:
class Solution1:
    """Cách 1: thử từng giá trị x trong khoảng [-B, B]."""
    def solveEquation(self, equation: str) -> str:
        def eval_side(side, x):
            total, sign, i, n = 0, 1, 0, len(side)
            while i < n:
                c = side[i]
                if c == "+":
                    sign, i = 1, i + 1
                elif c == "-":
                    sign, i = -1, i + 1
                else:
                    j = i
                    while j < n and side[j].isdigit():
                        j += 1
                    if j < n and side[j] == "x":
                        total += sign * (int(side[i:j]) if j > i else 1) * x
                        j += 1
                    else:
                        total += sign * int(side[i:j])
                    i, sign = j, 1
            return total

        left, right = equation.split("=")
        B = sum(int(t) for t in re.findall(r"\d+", equation)) + 1
        roots = [x for x in range(-B, B + 1) if eval_side(left, x) == eval_side(right, x)]
        if not roots:
            return "No solution"
        if len(roots) == 2 * B + 1:
            return "Infinite solutions"
        return f"x={roots[0]}"

for eq, expected in examples:
    assert Solution1().solveEquation(eq) == expected
print("cách 1 qua các ví dụ")

cách 1 qua các ví dụ


**Độ phức tạp.** Có `2B + 1` giá trị `x` để thử, mỗi lần tính hai vế tốn `O(L)` (`L` = độ dài chuỗi): **Time `O(B * L)`** với `B` có thể lên tới cỡ `25.000` khi `L = 1000` -> khoảng `50.000 * 1000 = 5*10^7` bước, rất chậm. Space `O(L)`.

**Nút thắt.** Ta tính lại **cả chuỗi** cho mỗi `x` thử. Nhưng một vế chỉ là `a*x + b`; chỉ cần đọc chuỗi **một lần** để lấy `(a, b)`. Đo thử để thấy độ chậm tăng theo độ dài:

In [12]:
def make_long(pairs):
    """Phương trình dài: base + nhiều cặp '+cx-cx' / '+c-c' làm đệm."""
    left = "x+5-3+x" + "".join(f"+{c}-{c}" for c in range(1, pairs + 1))
    return left + "=6+x-2"

print("độ dài | brute (ms)")
for pairs in [10, 20, 40]:
    eq = make_long(pairs)
    t0 = time.perf_counter()
    r = solve_brute(eq)
    print(f"{len(eq):>6} | {(time.perf_counter() - t0) * 1000:>9.1f}   -> {r}")

độ dài | brute (ms)
    55 |       3.1   -> x=2
   115 |      21.0   -> x=2


   235 |     194.8   -> x=2


`B` (khoảng thử) và độ dài `L` **cùng tăng** khi thêm số, nên thời gian tăng rất nhanh (gấp đôi độ dài thì chậm hơn nhiều lần). Cần một cách chỉ đọc chuỗi một lần.

## 6. Ý tưởng thuật toán / bước đột phá

**Mọi vế đều là hàm tuyến tính của `x`:** `vế = a*x + b`. Nên một lượt đọc chuỗi, cộng dồn hai số: `a` (tổng hệ số của `x`) và `b` (tổng hằng số). Với phương trình `a1*x + b1 = a2*x + b2`:

```
(a1 - a2) * x = b2 - b1
```

- `a1 != a2` -> `x = (b2 - b1) / (a1 - a2)` (nguyên theo đề, chia nguyên `//`).
- `a1 == a2` và `b1 == b2` -> **vô số nghiệm**.
- `a1 == a2` và `b1 != b2` -> **vô nghiệm**.

Ta có nhiều cách hiện thực ý tưởng "đưa về `a*x + b`":

- Cách 2: duyệt **từng vế**, mỗi vế ra `(a, b)`, rồi so sánh hai vế.
- Cách 3: duyệt **cả chuỗi một lượt**, số hạng ở vế phải bị **đổi dấu** (chuyển vế ngay khi đọc) -> chỉ còn `a*x + b = 0`.
- Cách 4: **không tự tách hệ số**, chỉ tính giá trị biểu thức tại `x = 0` và `x = 1` (hàm tuyến tính xác định bởi hai điểm).
- Cách 5: thay `x` bằng **số phức `i`**: phần ảo = hệ số, phần thực = hằng số (cho Python tự "tách").

## 7. Các cách tiếp theo

### Cách 2 — Duyệt từng vế, so sánh `(a1, b1)` với `(a2, b2)`

**Giải thích từng đoạn.**

1. `left, right = equation.split("=")`.
2. Hàm `parse(side)` trả về `(coef, const)`: duyệt chuỗi bằng `i`, giữ `sign`; gặp `+`/`-` cập nhật `sign`; gặp số hạng thì đọc chữ số, nếu sau đó là `x` thì cộng `sign * hệ số` vào `coef` (hệ số `1` nếu không có chữ số), ngược lại cộng `sign * số` vào `const`.
3. So sánh `a1, a2`: bằng nhau thì xét `b1 == b2` để chọn "Infinite"/"No solution"; khác nhau thì `x = (b2 - b1) // (a1 - a2)`.

Demo `parse` từng bước cho một vế:

In [13]:
def parse(side):
    coef = const = 0
    sign, i, n = 1, 0, len(side)
    while i < n:
        c = side[i]
        if c == "+":
            sign, i = 1, i + 1
        elif c == "-":
            sign, i = -1, i + 1
        else:
            j = i
            while j < n and side[j].isdigit():
                j += 1
            if j < n and side[j] == "x":
                coef += sign * (int(side[i:j]) if j > i else 1)
                j += 1
            else:
                const += sign * int(side[i:j])
            i, sign = j, 1
    return coef, const

print(parse("x+5-3+x"), parse("6+x-2"), parse("-x"), parse("0x"), parse("-100"))

(2, 2) (1, 4) (-1, 0) (0, 0) (0, -100)


**Code nộp LeetCode (cách 2):**

In [14]:
class Solution2:
    """Cách 2: phân tích từng vế thành (hệ số x, hằng số)."""
    def solveEquation(self, equation: str) -> str:
        def parse(side):
            coef = const = 0
            sign, i, n = 1, 0, len(side)
            while i < n:
                c = side[i]
                if c == "+":
                    sign, i = 1, i + 1
                elif c == "-":
                    sign, i = -1, i + 1
                else:
                    j = i
                    while j < n and side[j].isdigit():
                        j += 1
                    if j < n and side[j] == "x":
                        coef += sign * (int(side[i:j]) if j > i else 1)
                        j += 1
                    else:
                        const += sign * int(side[i:j])
                    i, sign = j, 1
            return coef, const

        left, right = equation.split("=")
        a1, b1 = parse(left)
        a2, b2 = parse(right)
        if a1 == a2:
            return "Infinite solutions" if b1 == b2 else "No solution"
        return f"x={(b2 - b1) // (a1 - a2)}"

for eq, expected in examples:
    assert Solution2().solveEquation(eq) == expected
print("cách 2 qua các ví dụ")

cách 2 qua các ví dụ


**Độ phức tạp (cách 2).** Mỗi ký tự được đọc đúng một lần: **Time `O(L)`**, Space `O(L)` cho hai vế sau `split` (có thể coi `O(1)` phụ nếu duyệt theo chỉ số).

### Cách 3 — Một lượt duyệt, đổi dấu khi qua dấu `=` (khuyên dùng)

**Giải thích.**

- Giữ hai số `a`, `b` đại diện cho `a*x + b = 0` (đã chuyển **mọi thứ về vế trái**) và một biến `side` là `+1` ở vế trái, `-1` ở vế phải.
- Mỗi số hạng được cộng với hệ số dấu `k = side * sign`. Một số hạng ở vế phải khi chuyển sang trái **đổi dấu**: đó đúng là `side = -1`.
- Gặp `=`: `side = -1` (và `sign = 1` cho số hạng đầu tiên của vế phải).
- Cuối cùng áp dụng ba kết cục ở mục 3.1: `a == 0` -> xét `b`; ngược lại `x = -b // a`.

Cách này không cần `split` và không cần hàm phụ.

In [15]:
def demo_single_pass(equation):
    a = b = 0
    side, sign = 1, 1
    i, n = 0, len(equation)
    while i < n:
        c = equation[i]
        if c == "+":
            sign, i = 1, i + 1
        elif c == "-":
            sign, i = -1, i + 1
        elif c == "=":
            side, sign, i = -1, 1, i + 1
        else:
            j = i
            while j < n and equation[j].isdigit():
                j += 1
            k = side * sign
            if j < n and equation[j] == "x":
                a += k * (int(equation[i:j]) if j > i else 1)
                j += 1
            else:
                b += k * int(equation[i:j])
            i, sign = j, 1
    return a, b

print(demo_single_pass("x+5-3+x=6+x-2"))     # (1, -2) nghĩa là x - 2 = 0
print(demo_single_pass("x=x"), demo_single_pass("x+1=x"), demo_single_pass("2x=x"))

(1, -2)
(0, 0) (0, 1) (1, 0)


**Code nộp LeetCode (cách 3) - đây cũng là code chính ở mục 9:**

In [16]:
class Solution3:
    """Cách 3: một lượt duyệt, vế phải bị đổi dấu."""
    def solveEquation(self, equation: str) -> str:
        a = b = 0                       # a*x + b = 0
        side, sign = 1, 1               # side: +1 vế trái, -1 vế phải
        i, n = 0, len(equation)
        while i < n:
            c = equation[i]
            if c == "+":
                sign, i = 1, i + 1
            elif c == "-":
                sign, i = -1, i + 1
            elif c == "=":
                side, sign, i = -1, 1, i + 1
            else:
                j = i
                while j < n and equation[j].isdigit():
                    j += 1
                k = side * sign
                if j < n and equation[j] == "x":
                    a += k * (int(equation[i:j]) if j > i else 1)
                    j += 1
                else:
                    b += k * int(equation[i:j])
                i, sign = j, 1
        if a == 0:
            return "Infinite solutions" if b == 0 else "No solution"
        return f"x={-b // a}"

for eq, expected in examples:
    assert Solution3().solveEquation(eq) == expected
print("cách 3 qua các ví dụ")

cách 3 qua các ví dụ


**Độ phức tạp (cách 3).** **Time `O(L)`**, Space `O(1)` (chỉ vài biến số nguyên).

### Cách 4 — Nội suy tuyến tính từ hai điểm `f(0)` và `f(1)`

**Giải thích.**

- Đặt `f(x) = (vế trái) - (vế phải)`, một hàm **tuyến tính** `a*x + b`. Cần giải `f(x) = 0`.
- `b = f(0)` và `a = f(1) - f(0)`.
- Để tính `f` mà **không tự phân tích chuỗi**, ta dùng `eval`: chèn dấu nhân `*` giữa chữ số và `x` bằng regex (`2x` -> `2*x`; `x` đứng một mình giữ nguyên), rồi `eval` với `x` là số cụ thể.
- Kiểm tra ký tự trước khi `eval` (chỉ gồm chữ số, `x`, `+`, `-`, `=`) và tắt các hàm dựng sẵn.

In [17]:
def make_f(equation):
    assert re.fullmatch(r"[0-9x+\-=]+", equation)          # an toàn: chỉ các ký tự cho phép
    left, right = (re.sub(r"(?<=\d)x", "*x", s) for s in equation.split("="))
    print("biểu thức sau khi chèn '*':", left, "|", right)
    return lambda x: eval(left, {"__builtins__": {}}, {"x": x}) - eval(right, {"__builtins__": {}}, {"x": x})

f = make_f("x+5-3+x=6+x-2")
print("f(0) =", f(0), "| f(1) =", f(1), "-> a =", f(1) - f(0), ", b =", f(0))

biểu thức sau khi chèn '*': x+5-3+x | 6+x-2
f(0) = -2 | f(1) = -1 -> a = 1 , b = -2


**Code nộp LeetCode (cách 4):**

In [18]:
class Solution4:
    """Cách 4: tính f(0), f(1) bằng eval để suy ra a và b."""
    def solveEquation(self, equation: str) -> str:
        left, right = (re.sub(r"(?<=\d)x", "*x", s) for s in equation.split("="))
        env = {"__builtins__": {}}

        def f(x):
            return eval(left, env, {"x": x}) - eval(right, env, {"x": x})

        b = f(0)
        a = f(1) - b
        if a == 0:
            return "Infinite solutions" if b == 0 else "No solution"
        return f"x={-b // a}"

for eq, expected in examples:
    assert Solution4().solveEquation(eq) == expected
print("cách 4 qua các ví dụ")

cách 4 qua các ví dụ


**Độ phức tạp (cách 4).** Hai lần `eval` mỗi lần `O(L)` -> **Time `O(L)`**, Space `O(L)`. Gọn nhưng phụ thuộc `eval` (không nên dùng với dữ liệu chưa kiểm tra) và chậm hơn các cách tự duyệt do chi phí biên dịch biểu thức.

### Cách 5 — Thay `x` bằng số phức `i`: phần ảo là hệ số, phần thực là hằng số

**Giải thích.**

- Trong Python, `1j` là đơn vị ảo. Thay mỗi chữ `x` bằng `j` rồi `eval` với `j = 1j`: `2x` thành `2j` (hằng số ảo `2`), `x` thành `j`, `5` vẫn là `5`.
- Cộng/trừ số phức cộng/trừ **riêng** phần thực và phần ảo, nên kết quả của một vế là `hằng số + hệ_số*j`: `.real` là hằng số, `.imag` là hệ số của `x`.
- Lấy `z = (vế trái) - (vế phải)`: `a = z.imag`, `b = z.real`. Mọi thứ còn lại như cũ.

In [19]:
env = {"__builtins__": {}}
for eq in ["x+5-3+x=6+x-2", "x=x", "2x=x", "-x=-1"]:
    left, right = eq.split("=")
    zl = eval(left.replace("x", "j"), env, {"j": 1j})
    zr = eval(right.replace("x", "j"), env, {"j": 1j})
    z = zl - zr
    print(f"{eq:16} vế trái = {zl}, vế phải = {zr} | z = {z} -> a = {z.imag}, b = {z.real}")

x+5-3+x=6+x-2    vế trái = (2+2j), vế phải = (4+1j) | z = (-2+1j) -> a = 1.0, b = -2.0
x=x              vế trái = 1j, vế phải = 1j | z = 0j -> a = 0.0, b = 0.0
2x=x             vế trái = 2j, vế phải = 1j | z = 1j -> a = 1.0, b = 0.0
-x=-1            vế trái = (-0-1j), vế phải = -1 | z = (1-1j) -> a = -1.0, b = 1.0


**Code nộp LeetCode (cách 5):**

In [20]:
class Solution5:
    """Cách 5: thay x bằng số phức 1j, phần ảo = hệ số, phần thực = hằng số."""
    def solveEquation(self, equation: str) -> str:
        left, right = equation.split("=")
        env = {"__builtins__": {}}
        z = eval(left.replace("x", "j"), env, {"j": 1j}) - eval(right.replace("x", "j"), env, {"j": 1j})
        a, b = int(z.imag), int(z.real)
        if a == 0:
            return "Infinite solutions" if b == 0 else "No solution"
        return f"x={-b // a}"

for eq, expected in examples:
    assert Solution5().solveEquation(eq) == expected
print("cách 5 qua các ví dụ")

cách 5 qua các ví dụ


**Độ phức tạp (cách 5).** **Time `O(L)`**, Space `O(L)`. Với hệ số/hằng số nhỏ (tổng cỡ vài chục nghìn), số thực dấu phẩy động biểu diễn **chính xác** các số nguyên nên `int(...)` an toàn. Cẩn thận: nếu một vế không chứa `x` thì `eval` trả về số **nguyên** (có `.real`, `.imag = 0` nên vẫn đúng).

**Tóm tắt năm cách:** (1) thử từng `x` - `O(B*L)`; (2) phân tích từng vế - `O(L)`; (3) một lượt, đổi dấu qua `=` - `O(L)`, `O(1)` bộ nhớ; (4) nội suy hai điểm bằng `eval`; (5) số phức `1j`.

## 8. TRACE từng bước

### 8.1. Trace cách 3 trên `"x+5-3+x=6+x-2"`

Mỗi dòng là một số hạng: vị trí, `side`, `sign`, loại số hạng, phần đóng góp và `(a, b)` sau bước đó (`a*x + b = 0`).

| Cột | Ý nghĩa |
|---|---|
| `side` | `+1` vế trái, `-1` vế phải |
| `k = side*sign` | dấu thật của số hạng sau khi chuyển hết về trái |
| `+a`, `+b` | phần cộng vào hệ số / hằng số |

In [21]:
def trace_single_pass(equation):
    a = b = 0
    side, sign = 1, 1
    i, n = 0, len(equation)
    while i < n:
        c = equation[i]
        if c == "+":
            sign, i = 1, i + 1
        elif c == "-":
            sign, i = -1, i + 1
        elif c == "=":
            side, sign, i = -1, 1, i + 1
            print(f"i={i - 1:>2} '=' -> chuyển sang vế phải: side = -1")
        else:
            start = i
            j = i
            while j < n and equation[j].isdigit():
                j += 1
            k = side * sign
            if j < n and equation[j] == "x":
                val = int(equation[i:j]) if j > i else 1
                a += k * val
                j += 1
                kind = f"hệ số x: +a = {k * val:+d}"
            else:
                val = int(equation[i:j])
                b += k * val
                kind = f"hằng số : +b = {k * val:+d}"
            print(f"i={start:>2} số hạng {equation[start:j]!r:>5} | side={side:+d} k={k:+d} | {kind:<22} | (a, b) = ({a}, {b})")
            i, sign = j, 1
    print(f"=> {a}*x + {b} = 0")
    if a == 0:
        return "Infinite solutions" if b == 0 else "No solution"
    return f"x={-b // a}"

print(trace_single_pass("x+5-3+x=6+x-2"))

i= 0 số hạng   'x' | side=+1 k=+1 | hệ số x: +a = +1       | (a, b) = (1, 0)
i= 2 số hạng   '5' | side=+1 k=+1 | hằng số : +b = +5      | (a, b) = (1, 5)
i= 4 số hạng   '3' | side=+1 k=-1 | hằng số : +b = -3      | (a, b) = (1, 2)
i= 6 số hạng   'x' | side=+1 k=+1 | hệ số x: +a = +1       | (a, b) = (2, 2)
i= 7 '=' -> chuyển sang vế phải: side = -1
i= 8 số hạng   '6' | side=-1 k=-1 | hằng số : +b = -6      | (a, b) = (2, -4)
i=10 số hạng   'x' | side=-1 k=-1 | hệ số x: +a = -1       | (a, b) = (1, -4)
i=12 số hạng   '2' | side=-1 k=+1 | hằng số : +b = +2      | (a, b) = (1, -2)
=> 1*x + -2 = 0
x=2


Quan sát: ở vế phải, số hạng `6` có `k = -1` nên cộng `-6` vào `b`; số hạng `-2` có `sign = -1`, `side = -1` nên `k = +1` (hai lần đổi dấu) và cộng `+2` vào `b`. Cuối cùng `a = 1`, `b = -2`: `x - 2 = 0` -> `x = 2`.

Thử thêm hai kết cục còn lại:

In [22]:
print(trace_single_pass("x=x"))
print(trace_single_pass("x+1=x"))

i= 0 số hạng   'x' | side=+1 k=+1 | hệ số x: +a = +1       | (a, b) = (1, 0)
i= 1 '=' -> chuyển sang vế phải: side = -1
i= 2 số hạng   'x' | side=-1 k=-1 | hệ số x: +a = -1       | (a, b) = (0, 0)
=> 0*x + 0 = 0
Infinite solutions
i= 0 số hạng   'x' | side=+1 k=+1 | hệ số x: +a = +1       | (a, b) = (1, 0)
i= 2 số hạng   '1' | side=+1 k=+1 | hằng số : +b = +1      | (a, b) = (1, 1)
i= 3 '=' -> chuyển sang vế phải: side = -1
i= 4 số hạng   'x' | side=-1 k=-1 | hệ số x: +a = -1       | (a, b) = (0, 1)
=> 0*x + 1 = 0
No solution


### 8.2. Trace cách 2 (từng vế)

In từng số hạng của mỗi vế và `(coef, const)` tích lũy.

In [23]:
def trace_parse(side_name, side):
    coef = const = 0
    sign, i, n = 1, 0, len(side)
    while i < n:
        c = side[i]
        if c == "+":
            sign, i = 1, i + 1
        elif c == "-":
            sign, i = -1, i + 1
        else:
            start = i
            j = i
            while j < n and side[j].isdigit():
                j += 1
            if j < n and side[j] == "x":
                coef += sign * (int(side[i:j]) if j > i else 1)
                j += 1
            else:
                const += sign * int(side[i:j])
            print(f"  {side_name}: số hạng {side[start:j]!r:>5}, dấu {sign:+d} -> (coef, const) = ({coef}, {const})")
            i, sign = j, 1
    return coef, const

eq = "x+5-3+x=6+x-2"
left, right = eq.split("=")
a1, b1 = trace_parse("trái", left)
a2, b2 = trace_parse("phải", right)
print(f"vế trái = {a1}x + {b1}; vế phải = {a2}x + {b2}")
print(f"(a1 - a2) x = b2 - b1  ->  {a1 - a2} x = {b2 - b1}  ->  x = {(b2 - b1) // (a1 - a2)}")

  trái: số hạng   'x', dấu +1 -> (coef, const) = (1, 0)
  trái: số hạng   '5', dấu +1 -> (coef, const) = (1, 5)
  trái: số hạng   '3', dấu -1 -> (coef, const) = (1, 2)
  trái: số hạng   'x', dấu +1 -> (coef, const) = (2, 2)
  phải: số hạng   '6', dấu +1 -> (coef, const) = (0, 6)
  phải: số hạng   'x', dấu +1 -> (coef, const) = (1, 6)
  phải: số hạng   '2', dấu -1 -> (coef, const) = (1, 4)
vế trái = 2x + 2; vế phải = 1x + 4
(a1 - a2) x = b2 - b1  ->  1 x = 2  ->  x = 2


### 8.3. Trace cách 1 (thử `x`), cách 4 (hai điểm) và cách 5 (số phức)

In [24]:
def trace_brute(equation, show=6):
    left, right = equation.split("=")
    B = sum(int(t) for t in re.findall(r"\d+", equation)) + 1
    print("B =", B, "(khoảng thử x từ", -B, "đến", B, ")")
    roots = []
    for x in range(-B, B + 1):
        lv, rv = eval_side(left, x), eval_side(right, x)
        if lv == rv:
            roots.append(x)
        if x < -B + show:
            print(f"  x={x:>3}: vế trái = {lv:>4}, vế phải = {rv:>4} {'<- bằng nhau' if lv == rv else ''}")
    print("  ...")
    print("các x làm hai vế bằng nhau:", roots)

trace_brute("x+5-3+x=6+x-2")

B = 17 (khoảng thử x từ -17 đến 17 )
  x=-17: vế trái =  -32, vế phải =  -13 
  x=-16: vế trái =  -30, vế phải =  -12 
  x=-15: vế trái =  -28, vế phải =  -11 
  x=-14: vế trái =  -26, vế phải =  -10 
  x=-13: vế trái =  -24, vế phải =   -9 
  x=-12: vế trái =  -22, vế phải =   -8 
  ...
các x làm hai vế bằng nhau: [2]


In [25]:
eq = "x+5-3+x=6+x-2"
print("cách 4:")
left, right = (re.sub(r"(?<=\d)x", "*x", s) for s in eq.split("="))
f = lambda x: eval(left, {"__builtins__": {}}, {"x": x}) - eval(right, {"__builtins__": {}}, {"x": x})
print(f"  f(0) = {f(0)} -> b = {f(0)}; f(1) = {f(1)} -> a = f(1) - f(0) = {f(1) - f(0)}; nghiệm x = {-f(0) // (f(1) - f(0))}")

print("cách 5:")
l, r = eq.split("=")
zl = eval(l.replace("x", "j"), {"__builtins__": {}}, {"j": 1j})
zr = eval(r.replace("x", "j"), {"__builtins__": {}}, {"j": 1j})
print(f"  vế trái = {zl}, vế phải = {zr}, hiệu = {zl - zr} -> a = {(zl - zr).imag}, b = {(zl - zr).real}")

cách 4:
  f(0) = -2 -> b = -2; f(1) = -1 -> a = f(1) - f(0) = 1; nghiệm x = 2
cách 5:
  vế trái = (2+2j), vế phải = (4+1j), hiệu = (-2+1j) -> a = 1.0, b = -2.0


Cả bốn cách cùng cho `a = 1`, `b = -2` (hoặc tương đương) và `x = 2`.

## 9. Code nộp LeetCode

Dùng **cách 3** (một lượt duyệt, không `split`, không `eval`):

```python
class Solution:
    def solveEquation(self, equation: str) -> str:
        a = b = 0                       # a*x + b = 0
        side, sign = 1, 1               # side: +1 vế trái, -1 vế phải
        i, n = 0, len(equation)
        while i < n:
            c = equation[i]
            if c == "+":
                sign, i = 1, i + 1
            elif c == "-":
                sign, i = -1, i + 1
            elif c == "=":
                side, sign, i = -1, 1, i + 1
            else:
                j = i
                while j < n and equation[j].isdigit():
                    j += 1
                k = side * sign
                if j < n and equation[j] == "x":
                    a += k * (int(equation[i:j]) if j > i else 1)
                    j += 1
                else:
                    b += k * int(equation[i:j])
                i, sign = j, 1
        if a == 0:
            return "Infinite solutions" if b == 0 else "No solution"
        return f"x={-b // a}"
```

Chạy lại đúng đoạn code trên:

In [26]:
class Solution:
    def solveEquation(self, equation: str) -> str:
        a = b = 0                       # a*x + b = 0
        side, sign = 1, 1               # side: +1 vế trái, -1 vế phải
        i, n = 0, len(equation)
        while i < n:
            c = equation[i]
            if c == "+":
                sign, i = 1, i + 1
            elif c == "-":
                sign, i = -1, i + 1
            elif c == "=":
                side, sign, i = -1, 1, i + 1
            else:
                j = i
                while j < n and equation[j].isdigit():
                    j += 1
                k = side * sign
                if j < n and equation[j] == "x":
                    a += k * (int(equation[i:j]) if j > i else 1)
                    j += 1
                else:
                    b += k * int(equation[i:j])
                i, sign = j, 1
        if a == 0:
            return "Infinite solutions" if b == 0 else "No solution"
        return f"x={-b // a}"

# Mô phỏng cách LeetCode gọi
for eq, expected in examples:
    got = Solution().solveEquation(eq)
    print(repr(eq), "->", repr(got), "OK" if got == expected else "SAI")
    assert got == expected

'x+5-3+x=6+x-2' -> 'x=2' OK
'x=x' -> 'Infinite solutions' OK
'2x=x' -> 'x=0' OK


## 10. Vì sao thuật toán đúng

**Mô hình.** Mỗi số hạng của phương trình là một trong hai dạng: `s * h * x` (hệ số `h >= 0`, dấu `s`) hoặc `s * c` (hằng số `c >= 0`). Vế trái là tổng các số hạng, vế phải cũng vậy. Phương trình `Trái = Phải` tương đương `Trái - Phải = 0`.

**Cộng dồn.** `Trái - Phải` là tổng của: mọi số hạng vế trái (giữ nguyên dấu) và mọi số hạng vế phải với **dấu đảo ngược**. Gọi `k = side * sign`: ở vế trái `side = +1` nên `k = sign`; ở vế phải `side = -1` nên `k = -sign`. Cộng `k * h` vào `a` cho mỗi số hạng chứa `x` và `k * c` vào `b` cho mỗi hằng số cho ta chính xác `Trái - Phải = a*x + b`.

**Đọc đúng số hạng.** Mỗi dấu `+`/`-` đặt `sign` cho số hạng **ngay sau nó**; sau khi đọc một số hạng, `sign` về `+1` (số hạng đầu của mỗi vế không có dấu thì dương; số hạng sau luôn có dấu tường minh nên `sign` được ghi đè). Dấu `=` đặt `side = -1` và `sign = 1`, nên số hạng đầu của vế phải (không dấu) là dương trước khi đảo vế. Hệ số `x` không có chữ số đứng trước nghĩa là `1`, và chữ số đứng trước `x` đọc đủ nhiều chữ số nên `0x` cho hệ số `0` và `100x` cho `100`.

**Kết luận theo `a`, `b`.** Giải `a*x + b = 0`:

- `a != 0`: `x = -b / a` duy nhất; đề đảm bảo là số nguyên nên `-b // a` chính xác (không làm tròn).
- `a == 0, b == 0`: `0 = 0` đúng với mọi `x` -> vô số nghiệm.
- `a == 0, b != 0`: `b = 0` sai với mọi `x` -> vô nghiệm.

**Các cách khác.** Cách 2 tính `(a1, b1)` và `(a2, b2)` rồi áp dụng `(a1 - a2) x = b2 - b1`, tương đương. Cách 1 liệt kê `x` trong `[-B, B]`: nghiệm duy nhất (nếu có) có `|x| = |b2 - b1| / |a1 - a2| <= |b2 - b1| <= tổng giá trị tuyệt đối các số < B` nên chắc chắn nằm trong khoảng thử; và hàm `Trái - Phải` tuyến tính chỉ có ba khả năng (không nghiệm / một nghiệm / mọi `x`). Cách 4 dùng `f(0)` và `f(1)` xác định `b` và `a` của hàm tuyến tính. Cách 5: phép cộng/trừ số phức tách riêng phần thực/ảo, và `n*j` có phần ảo `n`.

## 11. Phân tích Time / Space Complexity

Gọi `L = len(equation) <= 1000`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử từng `x` | `O(B * L)`, `B <= ~25.000` | `O(L)` | chậm khi phương trình dài |
| 2. Phân tích từng vế | `O(L)` | `O(L)` | `split` tạo hai chuỗi con |
| 3. Một lượt, đổi dấu | `O(L)` | `O(1)` | **dùng để nộp** |
| 4. Nội suy hai điểm (`eval`) | `O(L)` | `O(L)` | phụ thuộc `eval` |
| 5. Số phức `1j` (`eval`) | `O(L)` | `O(L)` | ngắn gọn, cần kiểm tra đầu vào |

Mọi cách tối ưu đều tuyến tính theo độ dài chuỗi: không thể nhanh hơn vì phải đọc từng ký tự. Cận `O(L)` với `L <= 1000` là rất nhỏ.

## 12. Bộ test

### 12.1. Ví dụ của đề và các phương trình tự đặt

Gom các cách vào một từ điển để chạy cùng bộ test:

In [27]:
METHODS = {
    "1-thu-x": Solution1().solveEquation,
    "2-tung-ve": Solution2().solveEquation,
    "3-mot-luot": Solution3().solveEquation,
    "4-hai-diem": Solution4().solveEquation,
    "5-so-phuc": Solution5().solveEquation,
    "nop": Solution().solveEquation,
}

def check_all(equation, expected, skip=()):
    for name, fn in METHODS.items():
        if name in skip:
            continue
        got = fn(equation)
        assert got == expected, (name, equation, got, expected)

for eq, expected in examples:
    check_all(eq, expected)
check_all("x+1=x", "No solution")
check_all("3x-2=x+8", "x=5")
print("ví dụ của đề và ví dụ tự đặt: OK")

ví dụ của đề và ví dụ tự đặt: OK


### 12.2. Edge case

- `x` đứng một mình hoặc có dấu: `x`, `-x` (hệ số `1` bị lược); hệ số `0` viết rõ: `0x`.
- Hai vế đổi chỗ, nghiệm `0`, nghiệm âm, nghiệm lớn (`100x=100`).
- Phương trình **không có `x`**: `1=1` (vô số), `1=2` (vô nghiệm).
- `x` chỉ ở một vế, chỉ ở vế phải, hoặc triệt tiêu cả hai vế.
- Dấu `-` đứng đầu vế phải: `x=-x`, `x=-5`.

In [28]:
edge = [
    ("x=x", "Infinite solutions"),
    ("x=-x", "x=0"),
    ("-x=x", "x=0"),
    ("-x=-x", "Infinite solutions"),
    ("x=5", "x=5"),
    ("5=x", "x=5"),
    ("x=-5", "x=-5"),
    ("-x=-1", "x=1"),
    ("0x=0", "Infinite solutions"),
    ("0x=1", "No solution"),
    ("x+1=x", "No solution"),
    ("1=1", "Infinite solutions"),
    ("1=2", "No solution"),
    ("0=0", "Infinite solutions"),
    ("2x=x", "x=0"),
    ("100x=100", "x=1"),
    ("99x-99=0", "x=1"),
    ("3x=-6", "x=-2"),
    ("-5x=10", "x=-2"),
    ("x-100=-100", "x=0"),
    ("x+x+x=x+x+x", "Infinite solutions"),
    ("10x-10x=0", "Infinite solutions"),
    ("10x-10x=1", "No solution"),
    ("1x=1", "x=1"),
    ("-1x=1", "x=-1"),
    ("0x+0x=0-0", "Infinite solutions"),
    ("100x+100x+100x=300", "x=1"),
]
for eq, expected in edge:
    check_all(eq, expected)
    print(f"{eq:22} -> {expected}")
print("edge case: OK")

x=x                    -> Infinite solutions
x=-x                   -> x=0
-x=x                   -> x=0
-x=-x                  -> Infinite solutions
x=5                    -> x=5
5=x                    -> x=5
x=-5                   -> x=-5
-x=-1                  -> x=1
0x=0                   -> Infinite solutions
0x=1                   -> No solution
x+1=x                  -> No solution
1=1                    -> Infinite solutions
1=2                    -> No solution
0=0                    -> Infinite solutions
2x=x                   -> x=0
100x=100               -> x=1
99x-99=0               -> x=1
3x=-6                  -> x=-2
-5x=10                 -> x=-2
x-100=-100             -> x=0
x+x+x=x+x+x            -> Infinite solutions
10x-10x=0              -> Infinite solutions
10x-10x=1              -> No solution
1x=1                   -> x=1
-1x=1                  -> x=-1
0x+0x=0-0              -> Infinite solutions
100x+100x+100x=300     -> x=1
edge case: OK


### 12.3. Phản ví dụ: input làm lộ các lỗi quen thuộc

Các phương trình dưới đây bắt những lỗi ở mục 13: `x`/`-x` không hệ số, dấu `-` ở vế phải, kết quả `-0`, chia cho `0`, vô nghiệm so với vô số nghiệm.

In [29]:
tricky = [
    "x+5-3+x=6+x-2",         # nhiều x không hệ số
    "-x+5=2x-4",             # -x ở đầu: -3x = -9
    "x=-x+2",                # dấu - ở đầu vế phải
    "10-x=x",                # x ở cuối vế trái
    "5=5+x",                 # x chỉ ở vế phải, nghiệm 0 (không phải -0)
    "-3x=0",                 # a < 0, b = 0: kết quả phải là 0 chứ không phải -0
    "7x-7x+3=3",             # triệt tiêu hoàn toàn
]
for eq in tricky:
    # đáp án kiểm bằng nội suy từ hàm giá trị eval_side (một đường tính khác với các lời giải)
    left, right = eq.split("=")
    f = lambda x: eval_side(left, x) - eval_side(right, x)
    a, b = f(1) - f(0), f(0)
    assert a == 0 or b % a == 0                       # nghiệm nguyên theo đề
    expected = classify(a, b)
    check_all(eq, expected)
    print(f"{eq:16} -> {expected}")

x+5-3+x=6+x-2    -> x=2
-x+5=2x-4        -> x=3
x=-x+2           -> x=1
10-x=x           -> x=5
5=5+x            -> x=0
-3x=0            -> x=0
7x-7x+3=3        -> Infinite solutions


### 12.4. Đối chiếu ngẫu nhiên có "đáp án thật" từ bộ sinh (seed cố định)

Cách làm: **sinh phương trình từ danh sách số hạng** (ta biết ngay tổng hệ số và tổng hằng số mỗi vế mà không phân tích chuỗi), viết ra chuỗi, rồi đối chiếu mọi cách với đáp án tính từ danh sách đó. Như vậy "đáp án" độc lập với các hàm phân tích.

Bộ sinh thỏa ràng buộc: số hạng nguyên trong `[-100, 100]` (dấu thể hiện bằng `+`/`-`), hệ số `1` đôi khi bị lược, không có số 0 đứng đầu, đúng một dấu `=`.

In [30]:
def fmt_terms(terms):
    """terms: list (kind, v, omit). kind 'x' hoặc 'c'. Trả về chuỗi vế."""
    out = []
    for idx, (kind, v, omit) in enumerate(terms):
        mag = abs(v)
        if kind == "x":
            body = "x" if (omit and mag == 1) else f"{mag}x"
        else:
            body = str(mag)
        sign = "-" if v < 0 else "+"
        out.append((sign if (idx > 0 or v < 0) else "") + body)
    return "".join(out)

def gen_side(rng, k, xr, cr):
    terms = []
    for _ in range(k):
        if rng.random() < 0.5:
            terms.append(("x", rng.randint(-xr, xr), rng.random() < 0.5))
        else:
            terms.append(("c", rng.randint(-cr, cr), False))
    return terms

def totals(terms):
    return (sum(v for kind, v, _ in terms if kind == "x"),
            sum(v for kind, v, _ in terms if kind == "c"))

def truth_from_totals(a1, b1, a2, b2):
    A, B = a1 - a2, b2 - b1
    if A == 0:
        return "Infinite solutions" if B == 0 else "No solution"
    assert B % A == 0
    return f"x={B // A}"

def gen_equation(rng, xr, cr, kmax, mirror=False):
    """Sinh (chuỗi, đáp án). Từ chối mẫu có nghiệm không nguyên (đề không cho)."""
    while True:
        L = gen_side(rng, rng.randint(1, kmax), xr, cr)
        if mirror:      # vế phải có cùng tổng hệ số x với vế trái -> a1 == a2
            Rx = [t for t in L if t[0] == "x"]
            R = [("x", v, om) for _, v, om in rng.sample(Rx, len(Rx))] + gen_side(rng, rng.randint(0, 2), 0, cr)
            R = [t for t in R if t[0] == "x" or t[0] == "c"]
            if not R:
                continue
        else:
            R = gen_side(rng, rng.randint(1, kmax), xr, cr)
        a1, b1 = totals(L)
        a2, b2 = totals(R)
        A, B = a1 - a2, b2 - b1
        if A != 0 and B % A != 0:
            continue
        eq = fmt_terms(L) + "=" + fmt_terms(R)
        return eq, truth_from_totals(a1, b1, a2, b2)

rng = random.Random(640)
for _ in range(5):
    print(gen_equation(rng, 3, 9, 4))

('-4=-2x', 'x=2')
('0x-6+2x-2x=-2x+2x+3x', 'x=-2')
('-4-5=3x+6', 'x=-5')
('-3x=-2x+0', 'x=0')
('-x-3x+x+1x=2x', 'x=0')


In [31]:
rng = random.Random(6400)
counts = {"x": 0, "Infinite solutions": 0, "No solution": 0}
for _ in range(1500):
    mode = rng.choice(["small", "small", "big", "mirror"])
    if mode == "small":
        eq, exp = gen_equation(rng, 3, 9, 5)
    elif mode == "big":
        eq, exp = gen_equation(rng, 100, 100, 3)
    else:
        eq, exp = gen_equation(rng, 100, 100, 5, mirror=True)
    assert eq.count("=") == 1 and 3 <= len(eq) <= 1000
    # Cách 1 chỉ chạy khi B đủ nhỏ để không quá chậm
    big_B = sum(int(t) for t in re.findall(r"\d+", eq)) > 600
    check_all(eq, exp, skip=("1-thu-x",) if big_B else ())
    counts["x" if exp.startswith("x=") else exp] += 1
print("1500 phương trình ngẫu nhiên: OK |", counts)

1500 phương trình ngẫu nhiên: OK | {'x': 755, 'Infinite solutions': 49, 'No solution': 696}


### 12.5. Phương trình dài (gần `1000` ký tự) và hiệu năng

Lấy phương trình gốc có đáp án đã biết rồi **chèn các cặp triệt tiêu** (`+37x-37x`, `+58-58`) vào các vị trí ngẫu nhiên cho tới khi dài gần `1000` ký tự. Đáp án không đổi.

In [32]:
def pad_equation(base, rng, target_len=990):
    left, right = base.split("=")
    sides = [left, right]
    while len(sides[0]) + len(sides[1]) + 1 < target_len - 8:
        idx = rng.randint(0, 1)
        c = rng.randint(1, 100)
        pair = rng.choice([f"+{c}x-{c}x", f"+{c}-{c}", f"-{c}x+{c}x", f"-{c}+{c}"])
        s = sides[idx]
        cuts = [p for p in range(1, len(s)) if s[p] in "+-"] + [len(s)]
        p = rng.choice(cuts)
        sides[idx] = s[:p] + pair + s[p:]
    return sides[0] + "=" + sides[1]

rng = random.Random(1000)
for base, expected in [("x+5-3+x=6+x-2", "x=2"), ("x=x", "Infinite solutions"),
                       ("x+1=x", "No solution"), ("-x+7=3x-9", "x=4"), ("5x=-5x", "x=0")]:
    eq = pad_equation(base, rng)
    assert eq.count("=") == 1 and len(eq) <= 1000, len(eq)
    skip = ("1-thu-x",)        # cách 1 quá chậm trên phương trình dài (xem mục 5)
    check_all(eq, expected, skip=skip)
    print(f"{base:18} -> độ dài {len(eq):>4}: các cách đều cho {expected!r}")

x+5-3+x=6+x-2      -> độ dài  985: các cách đều cho 'x=2'
x=x                -> độ dài  989: các cách đều cho 'Infinite solutions'
x+1=x              -> độ dài  983: các cách đều cho 'No solution'
-x+7=3x-9          -> độ dài  987: các cách đều cho 'x=4'
5x=-5x             -> độ dài  984: các cách đều cho 'x=0'


In [33]:
eq = pad_equation("x+5-3+x=6+x-2", random.Random(5), 995)
print("độ dài:", len(eq))
for name, fn in METHODS.items():
    if name == "1-thu-x":
        continue
    t0 = time.perf_counter()
    for _ in range(200):
        r = fn(eq)
    print(f"{name:12} {(time.perf_counter() - t0) / 200 * 1000:7.3f} ms / lần ->", r)

độ dài: 987
2-tung-ve      0.156 ms / lần -> x=2
3-mot-luot     0.151 ms / lần -> x=2


4-hai-diem     0.617 ms / lần -> x=2


5-so-phuc      0.449 ms / lần -> x=2
nop            0.215 ms / lần -> x=2


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Coi `x` (hoặc `-x`) không có chữ số là hệ số `0`/lỗi

**Code sai:**

In [34]:
def wrong_coef(equation):
    a = b = 0
    side, sign = 1, 1
    i, n = 0, len(equation)
    while i < n:
        c = equation[i]
        if c == "+":
            sign, i = 1, i + 1
        elif c == "-":
            sign, i = -1, i + 1
        elif c == "=":
            side, sign, i = -1, 1, i + 1
        else:
            j = i
            while j < n and equation[j].isdigit():
                j += 1
            k = side * sign
            if j < n and equation[j] == "x":
                a += k * int(equation[i:j])          # SAI: "x" không có chữ số -> int("")
                j += 1
            else:
                b += k * int(equation[i:j])
            i, sign = j, 1
    return "Infinite solutions" if a == 0 and b == 0 else ("No solution" if a == 0 else f"x={-b // a}")

try:
    wrong_coef("x+5-3+x=6+x-2")
except ValueError as e:
    print("sai  : ValueError:", e)
print("đúng :", Solution().solveEquation("x+5-3+x=6+x-2"))

sai  : ValueError: invalid literal for int() with base 10: ''
đúng : x=2


**Vì sao sai:** đề lược hệ số `1`: `x` nghĩa là `1x`. `int("")` báo lỗi (hoặc nếu viết kiểu `int(...) or 0` thì ra `0` sai nghĩa).

**Code đúng:** `coef = int(equation[i:j]) if j > i else 1`.

### Lỗi 2: Quên đổi dấu khi chuyển vế

In [35]:
def wrong_no_flip(equation):
    a = b = 0
    sign, i, n = 1, 0, len(equation)
    while i < n:
        c = equation[i]
        if c == "+":
            sign, i = 1, i + 1
        elif c == "-":
            sign, i = -1, i + 1
        elif c == "=":
            sign, i = 1, i + 1                      # SAI: không đảo dấu cho vế phải
        else:
            j = i
            while j < n and equation[j].isdigit():
                j += 1
            if j < n and equation[j] == "x":
                a += sign * (int(equation[i:j]) if j > i else 1)
                j += 1
            else:
                b += sign * int(equation[i:j])
            i, sign = j, 1
    if a == 0:
        return "Infinite solutions" if b == 0 else "No solution"
    return f"x={-b // a}"

print("sai  :", wrong_no_flip("2x=x"), "|", wrong_no_flip("x=x"), "(cộng cả hai vế thay vì trừ)")
print("đúng :", Solution().solveEquation("2x=x"), "|", Solution().solveEquation("x=x"))

sai  : x=0 | x=0 (cộng cả hai vế thay vì trừ)
đúng : x=0 | Infinite solutions


**Vì sao sai:** phương trình `Trái = Phải` tương đương `Trái - Phải = 0`, nên số hạng vế phải phải **đổi dấu**. Cộng cả hai vế làm `x=x` thành `2x = 0` (nghiệm `0`) thay vì `0 = 0`.

**Code đúng:** dùng `side = -1` sau dấu `=` (nhân vào dấu số hạng), hoặc tính `(a1 - a2)` và `(b2 - b1)` ở cách 2.

### Lỗi 3: Chia số thực `/` (ra `"x=2.0"` hoặc `"x=-0.0"`)

In [36]:
def wrong_float(equation):
    left, right = equation.split("=")
    a1, b1 = parse(left)
    a2, b2 = parse(right)
    if a1 == a2:
        return "Infinite solutions" if b1 == b2 else "No solution"
    return f"x={(b2 - b1) / (a1 - a2)}"             # SAI: dùng / thay vì //

for eq in ["x+5-3+x=6+x-2", "2x=x", "5=5+x"]:
    print(f"{eq:16} sai: {wrong_float(eq)!r:10} đúng: {Solution().solveEquation(eq)!r}")

x+5-3+x=6+x-2    sai: 'x=2.0'    đúng: 'x=2'
2x=x             sai: 'x=0.0'    đúng: 'x=0'
5=5+x            sai: 'x=-0.0'   đúng: 'x=0'


**Vì sao sai:** `/` luôn trả về số thực, nên `"x=2.0"`, và `0 / -1` còn cho `-0.0` (`"x=-0.0"`). Đề muốn **số nguyên**.

**Code đúng:** `//` (đúng vì nghiệm nguyên nên phép chia hết), hoặc `int(...)` sau khi chia.

### Lỗi 4: Chia cho `0` / nhầm `No solution` với `Infinite solutions`

In [37]:
def wrong_divide_first(equation):
    left, right = equation.split("=")
    a1, b1 = parse(left)
    a2, b2 = parse(right)
    return f"x={(b2 - b1) // (a1 - a2)}"            # SAI: chưa kiểm tra a1 == a2

def wrong_mix_up(equation):
    left, right = equation.split("=")
    a1, b1 = parse(left)
    a2, b2 = parse(right)
    if a1 == a2:
        return "No solution"                         # SAI: a1 == a2 còn phải xét b1 == b2
    return f"x={(b2 - b1) // (a1 - a2)}"

try:
    wrong_divide_first("x=x")
except ZeroDivisionError as e:
    print("sai  : ZeroDivisionError:", e)
print("nhầm :", wrong_mix_up("x=x"), "(đúng phải là Infinite solutions)")
print("đúng :", Solution().solveEquation("x=x"), "|", Solution().solveEquation("x+1=x"))

sai  : ZeroDivisionError: integer division or modulo by zero
nhầm : No solution (đúng phải là Infinite solutions)
đúng : Infinite solutions | No solution


**Vì sao sai:** khi `a1 == a2` phép chia cho `0`. Hơn nữa hai trường hợp cùng `a1 == a2` có kết cục khác nhau: `b1 == b2` -> **vô số nghiệm**, `b1 != b2` -> **vô nghiệm**.

**Code đúng:** kiểm tra `a == 0` **trước** khi chia, rồi xét `b`.

### Lỗi 5: Tách chuỗi sai chỗ (`split("+")`)

In [38]:
def wrong_split_plus(equation):
    left, right = equation.split("=")
    def parse_bad(side):
        coef = const = 0
        for t in side.split("+"):                    # SAI: bỏ sót dấu '-' ở giữa token
            if t.endswith("x"):
                coef += int(t[:-1] or 1)
            else:
                const += int(t)
        return coef, const
    a1, b1 = parse_bad(left)
    a2, b2 = parse_bad(right)
    if a1 == a2:
        return "Infinite solutions" if b1 == b2 else "No solution"
    return f"x={(b2 - b1) // (a1 - a2)}"

print("đúng :", Solution().solveEquation("x+5-3+x=6+x-2"))
for eq in ["x+5-3+x=6+x-2", "2x-3=1"]:
    try:
        print("sai  :", eq, "->", wrong_split_plus(eq))
    except ValueError as e:
        print("sai  :", eq, "-> ValueError:", e)

đúng : x=2
sai  : x+5-3+x=6+x-2 -> ValueError: invalid literal for int() with base 10: '5-3'
sai  : 2x-3=1 -> ValueError: invalid literal for int() with base 10: '2x-3'


**Vì sao sai:** token `5-3` không phải một số; `int("5-3")` lỗi. Dấu `-` vừa là dấu trừ giữa hai số hạng vừa là dấu âm; phải xử lý **từng số hạng có dấu**.

**Code đúng:** duyệt ký tự với biến `sign` (như cách 2/3), hoặc dùng regex `[+-]?\d*x|[+-]?\d+` để lấy từng số hạng kèm dấu.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Nhận dạng:** "phương trình tuyến tính cho dạng chuỗi" -> quy về **`a*x + b`**. Hai số `(a, b)` mô tả toàn bộ một vế; mọi việc còn lại chỉ là đại số lớp 7.
- **Ba kết cục** chỉ phụ thuộc `(a, b)` sau chuyển vế: `a != 0` một nghiệm; `a == 0, b == 0` vô số; `a == 0, b != 0` vô nghiệm. Viết hàm `classify(a, b)` nhỏ và tin tưởng nó.
- **Chuyển vế bằng cách đổi dấu** khi đọc: dùng biến `side = ±1` thay vì `split` (cách 3).
- Hệ số **lược bỏ**: `x`, `-x`; hệ số `0` viết rõ `0x`; kiểm tra `j > i` để biết có chữ số hay không.
- Chia hết nên dùng `//`; luôn kiểm `a == 0` **trước** khi chia.
- **Mẹo thay biến bằng số phức** (cách 5) cho gọn khi ngôn ngữ hỗ trợ; nhớ kiểm tra đầu vào trước `eval`.
- **Kiểm tra nghiệm:** thay `x` tìm được vào hai vế (hàm `eval_side`) phải ra hai giá trị bằng nhau.
- Bài họ hàng: LeetCode 592 (Fraction Addition and Subtraction), 224/227 (Basic Calculator), 537 (Complex Number Multiplication), 150 (Evaluate Reverse Polish Notation), 8 (String to Integer).

## 15. Biến thể

### 15.1. Nghiệm không nguyên: trả về phân số `"x=p/q"`

Nếu đề không đảm bảo nghiệm nguyên, dùng `Fraction` để giữ độ chính xác.

In [39]:
def solve_fraction(equation):
    # lấy (a, b) từ cách 3 bằng cùng đoạn duyệt
    a, b = demo_single_pass(equation)
    if a == 0:
        return "Infinite solutions" if b == 0 else "No solution"
    x = Fraction(-b, a)
    return f"x={x.numerator}" if x.denominator == 1 else f"x={x.numerator}/{x.denominator}"

for eq in ["2x=1", "3x=-4", "6x=3x+9", "x=x+1", "4x=-8"]:
    print(f"{eq:10} -> {solve_fraction(eq)}")
assert solve_fraction("2x=1") == "x=1/2" and solve_fraction("4x=-8") == "x=-2"

2x=1       -> x=1/2
3x=-4      -> x=-4/3
6x=3x+9    -> x=3
x=x+1      -> No solution
4x=-8      -> x=-2


### 15.2. Có dấu ngoặc: `2(x+1)=x+5`

Với số phức (cách 5) việc mở rộng khá dễ: chèn dấu `*` cho phép nhân ngầm giữa số và `(`, rồi để Python tính. Cần kiểm tra ký tự cho phép (ở đây thêm `(` và `)`).

In [40]:
def solve_with_parens(equation):
    assert re.fullmatch(r"[0-9x+\-=()]+", equation)
    left, right = equation.split("=")
    def val(s):
        s = re.sub(r"(?<=\d)(?=\()", "*", s)            # 2( -> 2*(
        s = re.sub(r"(?<=\))(?=[\dx(])", "*", s)         # )2, )x, )( -> )*...
        s = re.sub(r"(?<=\d)x", "j", s)                  # 2x -> 2j
        s = s.replace("x", "j")                           # x  -> j
        return eval(s, {"__builtins__": {}}, {"j": 1j})
    z = val(left) - val(right)
    a, b = int(z.imag), int(z.real)
    return "Infinite solutions" if a == 0 and b == 0 else ("No solution" if a == 0 else f"x={-b // a}")

for eq in ["2(x+1)=x+5", "3(x-2)=x+4", "2(x+1)=2x+2", "(x+1)(2)=2x+3", "-(x)=-4"]:
    try:
        print(f"{eq:16} -> {solve_with_parens(eq)}")
    except Exception as e:
        print(f"{eq:16} -> (không hỗ trợ: {type(e).__name__})")

2(x+1)=x+5       -> x=3
3(x-2)=x+4       -> x=5
2(x+1)=2x+2      -> Infinite solutions
(x+1)(2)=2x+3    -> No solution
-(x)=-4          -> x=4


Ghi chú: `"-(x)=-4"` hoạt động vì `eval` hiểu `-(j)`; còn `"(x+1)(2)"` cần quy tắc nhân ngầm `)(`: đã được chèn bởi `re.sub` thứ hai. Với các biểu thức phức tạp hơn nên viết bộ phân tích cú pháp riêng (xem bài Basic Calculator).

### 15.3. Biến khác tên `x`

Tham số hóa tên biến: `solve_var(equation, var="y")`. Chỉ cần đổi ký tự so sánh ở cách 3.

In [41]:
def solve_var(equation, var="x"):
    a = b = 0
    side, sign = 1, 1
    i, n = 0, len(equation)
    while i < n:
        c = equation[i]
        if c == "+":
            sign, i = 1, i + 1
        elif c == "-":
            sign, i = -1, i + 1
        elif c == "=":
            side, sign, i = -1, 1, i + 1
        else:
            j = i
            while j < n and equation[j].isdigit():
                j += 1
            k = side * sign
            if j < n and equation[j] == var:
                a += k * (int(equation[i:j]) if j > i else 1)
                j += 1
            else:
                b += k * int(equation[i:j])
            i, sign = j, 1
    if a == 0:
        return "Infinite solutions" if b == 0 else "No solution"
    return f"{var}={-b // a}"

print(solve_var("3y+1=y+9", "y"), solve_var("t=t", "t"), solve_var("2x=4"))

y=4 Infinite solutions x=2


### 15.4. Chỉ tính giá trị biểu thức tại `x` cho trước

Đó chính là `eval_side` ở mục 5; ý nghĩa: **giải phương trình** và **kiểm tra nghiệm** là hai bài toán đối nghịch.

In [42]:
def check_solution(equation, x):
    left, right = equation.split("=")
    return eval_side(left, x) == eval_side(right, x)

for eq, expected in examples:
    if expected.startswith("x="):
        sol = int(expected[2:])
        print(eq, "-> x =", sol, "thỏa?", check_solution(eq, sol), "| x =", sol + 1, "thỏa?", check_solution(eq, sol + 1))
        assert check_solution(eq, sol) and not check_solution(eq, sol + 1)

x+5-3+x=6+x-2 -> x = 2 thỏa? True | x = 3 thỏa? False
2x=x -> x = 0 thỏa? True | x = 1 thỏa? False


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử từng `x` | tính hai vế cho mọi `x` trong `[-B, B]` | `O(B * L)` | `O(L)` | chỉ để minh họa/kiểm chứng |
| 2. Phân tích từng vế | `(a1, b1)` và `(a2, b2)` rồi so sánh | `O(L)` | `O(L)` | muốn code dễ đọc, hai bước rõ ràng |
| 3. Một lượt, đổi dấu | `side = ±1`, gom về `a*x + b = 0` | `O(L)` | `O(1)` | **nộp bài** |
| 4. Nội suy hai điểm | `a = f(1) - f(0)`, `b = f(0)` bằng `eval` | `O(L)` | `O(L)` | khi có sẵn bộ tính biểu thức |
| 5. Số phức `1j` | thực = hằng số, ảo = hệ số | `O(L)` | `O(L)` | ngắn gọn, thích hợp prototype |

Kết luận: nộp **cách 3**. Cách 2 dễ hiểu nhất; cách 4 và 5 cho thấy hai "mẹo" đại số (hàm tuyến tính xác định bởi hai điểm; số phức tách hai loại hạng tử) nhưng cần `eval`, nên cẩn trọng với đầu vào.

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán đầu ra rồi chạy:

In [43]:
print(Solution().solveEquation("x=x"))
print(Solution().solveEquation("2x=x"))
print(Solution().solveEquation("x+1=x"))
print(Solution().solveEquation("-x=2x-9"))

Infinite solutions
x=0
No solution
x=3


**Bài 2 (tính tay).** Giải `3x-2=x+8` bằng cách tính `(a1, b1)`, `(a2, b2)` rồi gán đáp án (chuỗi) vào `ans2` (đang là `None`).

In [44]:
ans2 = None      # TODO: điền chuỗi, ví dụ "x=7"

**Bài 3 (điền TODO).** Viết `split_terms(side)` trả về **list các số hạng có dấu** của một vế, theo thứ tự. Ví dụ `split_terms("x+5-3+x")` -> `["x", "+5", "-3", "+x"]`; `split_terms("-x+0x-10")` -> `["-x", "+0x", "-10"]`. (Gợi ý: số hạng bắt đầu ở mỗi dấu `+`/`-`, hoặc ở đầu chuỗi.)

In [45]:
def split_terms(side):
    # TODO
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_solve(equation)` theo cách 3 (một lượt, `side = ±1`).

In [46]:
def my_solve(equation):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) Viết `solve_fraction_str(equation)` trả về `"x=p/q"` (phân số tối giản, mẫu dương) nếu nghiệm không nguyên, `"x=p"` nếu nguyên, hoặc hai kết cục còn lại. (b) Viết `check_solution(equation, x)` trả về `True` nếu thay `x` vào cho hai vế bằng nhau. Gợi ý (a): dùng `fractions.Fraction`; (b): tính giá trị từng vế.

In [47]:
def solve_fraction_str(equation):
    # TODO
    pass

def check_solution_mine(equation, x):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: dùng ba kết cục ở mục 3.1; `-x=2x-9` -> `-3x = -9`.
- Bài 2: vế trái `3x - 2`: `a1 = 3, b1 = -2`; vế phải `x + 8`: `a2 = 1, b2 = 8`.
- Bài 3: duyệt từng ký tự; bắt đầu số hạng mới khi gặp `+` hoặc `-` (trừ vị trí đầu).
- Bài 4: nhớ `side`, `sign`, đọc chữ số rồi kiểm tra có `x` không.
- Bài 5: `Fraction(-b, a)` đã tự tối giản và đưa dấu về tử số.

**Mức 2 (vừa):**
- Bài 2: `(a1 - a2) x = b2 - b1` -> `2x = 10`.
- Bài 3: duy trì `current = ""`; gặp `+`/`-` thì đẩy `current` (nếu khác rỗng) vào kết quả và bắt đầu số hạng mới với dấu đó.
- Bài 4: `k = side * sign`; số hạng có `x`: `a += k * (hệ số hoặc 1)`.
- Bài 5(b): tái sử dụng hàm tính giá trị một vế với `x` cụ thể (như `eval_side` ở mục 5).

**Mức 3 (gần đáp án):**
- Bài 2: `x=5`.
- Bài 3: sau vòng lặp nhớ đẩy `current` cuối cùng vào kết quả.
- Bài 5(a): `x = Fraction(-b, a)`; nếu `x.denominator == 1` in `x.numerator`, ngược lại `f"{x.numerator}/{x.denominator}"`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [48]:
# Bài 1: Infinite solutions, x=0, No solution, x=3
assert Solution().solveEquation("x=x") == "Infinite solutions"
assert Solution().solveEquation("2x=x") == "x=0"
assert Solution().solveEquation("x+1=x") == "No solution"
assert Solution().solveEquation("-x=2x-9") == "x=3"

# Bài 2: 3x-2 = x+8 -> (3-1)x = 8+2 -> x = 5
ans2 = "x=5"
assert ans2 == Solution().solveEquation("3x-2=x+8")
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [49]:
# Bài 3
def split_terms(side):
    out, current = [], ""
    for c in side:
        if c in "+-" and current:
            out.append(current)
            current = c
        else:
            current += c
    if current:
        out.append(current)
    return out

assert split_terms("x+5-3+x") == ["x", "+5", "-3", "+x"]
assert split_terms("-x+0x-10") == ["-x", "+0x", "-10"]
assert split_terms("7") == ["7"]
assert split_terms("100x") == ["100x"]
# nối lại các số hạng phải ra đúng vế ban đầu
rng = random.Random(3)
for _ in range(100):
    eq, _ = gen_equation(rng, 100, 100, 5)
    for side in eq.split("="):
        assert "".join(split_terms(side)) == side
print("Bài 3 OK")

Bài 3 OK


In [50]:
# Bài 4
def my_solve(equation):
    a = b = 0
    side, sign = 1, 1
    i, n = 0, len(equation)
    while i < n:
        c = equation[i]
        if c == "+":
            sign, i = 1, i + 1
        elif c == "-":
            sign, i = -1, i + 1
        elif c == "=":
            side, sign, i = -1, 1, i + 1
        else:
            j = i
            while j < n and equation[j].isdigit():
                j += 1
            k = side * sign
            if j < n and equation[j] == "x":
                a += k * (int(equation[i:j]) if j > i else 1)
                j += 1
            else:
                b += k * int(equation[i:j])
            i, sign = j, 1
    if a == 0:
        return "Infinite solutions" if b == 0 else "No solution"
    return f"x={-b // a}"

rng = random.Random(4)
for _ in range(500):
    eq, exp = gen_equation(rng, rng.choice([3, 100]), rng.choice([9, 100]), 4)
    assert my_solve(eq) == exp, (eq, my_solve(eq), exp)
print("Bài 4 OK")

Bài 4 OK


In [51]:
# Bài 5
def solve_fraction_str(equation):
    left, right = equation.split("=")
    a1, b1 = parse(left)
    a2, b2 = parse(right)
    A, B = a1 - a2, b2 - b1
    if A == 0:
        return "Infinite solutions" if B == 0 else "No solution"
    x = Fraction(B, A)
    return f"x={x.numerator}" if x.denominator == 1 else f"x={x.numerator}/{x.denominator}"

def check_solution_mine(equation, x):
    left, right = equation.split("=")
    return eval_side(left, x) == eval_side(right, x)

assert solve_fraction_str("2x=1") == "x=1/2"
assert solve_fraction_str("3x=-4") == "x=-4/3"
assert solve_fraction_str("-2x=1") == "x=-1/2"
assert solve_fraction_str("6x=3x+9") == "x=3"
assert solve_fraction_str("x=x") == "Infinite solutions"
assert solve_fraction_str("x+1=x") == "No solution"
assert check_solution_mine("x+5-3+x=6+x-2", 2) and not check_solution_mine("x+5-3+x=6+x-2", 3)
rng = random.Random(5)
for _ in range(300):
    eq, exp = gen_equation(rng, 100, 100, 4)
    if exp.startswith("x="):
        assert check_solution_mine(eq, int(exp[2:]))
        assert solve_fraction_str(eq) == exp
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (một lượt, đổi dấu khi qua dấu `=`):**

```python
class Solution:
    def solveEquation(self, equation: str) -> str:
        a = b = 0                       # a*x + b = 0
        side, sign = 1, 1               # side: +1 vế trái, -1 vế phải
        i, n = 0, len(equation)
        while i < n:
            c = equation[i]
            if c == "+":   sign, i = 1, i + 1
            elif c == "-": sign, i = -1, i + 1
            elif c == "=": side, sign, i = -1, 1, i + 1
            else:
                j = i
                while j < n and equation[j].isdigit(): j += 1
                k = side * sign
                if j < n and equation[j] == "x":
                    a += k * (int(equation[i:j]) if j > i else 1)   # "x" = 1x
                    j += 1
                else:
                    b += k * int(equation[i:j])
                i, sign = j, 1
        if a == 0:
            return "Infinite solutions" if b == 0 else "No solution"
        return f"x={-b // a}"
```

**Checklist trước khi submit:**

- [ ] `x` và `-x` (không chữ số) được hiểu là hệ số `1` và `-1`; `0x` là hệ số `0`?
- [ ] Số hạng ở **vế phải** đã **đổi dấu** (`side = -1`)?
- [ ] Dấu `-` ở đầu mỗi vế (kể cả ngay sau `=`) xử lý đúng?
- [ ] Kiểm tra `a == 0` **trước** khi chia; phân biệt `b == 0` (vô số) và `b != 0` (vô nghiệm)?
- [ ] Dùng `//` (hoặc số nguyên), không in `"x=2.0"` / `"x=-0.0"`?
- [ ] Đã thử: `x=x`, `0x=0`, `0x=1`, `x=-x`, `-x=-1`, `1=1`, `1=2`, `100x=100`, nghiệm âm, nghiệm `0`?
- [ ] Không dùng `eval` trên dữ liệu chưa kiểm tra (nếu dùng cách 4/5, đã lọc ký tự)?